# PitWall Intelligence — Notebook 01: ETL Pipeline
**Sprint 1 | Data Acquisition & EDA**

This notebook pulls F1 historical data from the **Jolpica-F1 API** (a free, open Ergast mirror)
and stores it in a structured SQLite database on Google Drive.

**Focal era: 2014–2024** (V6 hybrid era — sponsorship comparisons are era-consistent)

Tables created:
- `races` — every Grand Prix per season
- `results` — finishing positions, points, status
- `constructors` — team metadata
- `drivers` — driver metadata
- `qualifying` — Q1/Q2/Q3 lap times
- `constructor_standings` — end-of-season standings
- `driver_standings` — end-of-season standings

**Runtime estimate: ~70–90 min on home WiFi (rate-limited to 1 req/sec)**

In [1]:
# ── Cell 1: Mount Drive & create folder structure ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/PitWall'
DATA_DIR     = f'{PROJECT_ROOT}/data'
RAW_DIR      = f'{DATA_DIR}/raw'
EXPORTS_DIR  = f'{DATA_DIR}/exports'
DB_PATH      = f'{DATA_DIR}/pitwall.db'

for d in [DATA_DIR, RAW_DIR, EXPORTS_DIR,
          f'{PROJECT_ROOT}/notebooks', f'{PROJECT_ROOT}/reports']:
    os.makedirs(d, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Database:     {DB_PATH}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/PitWall
Database:     /content/drive/MyDrive/PitWall/data/pitwall.db


In [2]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
!pip install -q requests tenacity tqdm pandas

In [13]:
# ── Cell 3: Rate-limited Jolpica API client ─────────────────────────────────
import requests, time, json, os
from pathlib import Path
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

BASE_URL   = 'https://api.jolpi.ca/ergast/f1'
THROTTLE_S = 1.1          # seconds between requests (stay well under 500/hr)
_last_call = [0.0]

@retry(
    retry=retry_if_exception_type(requests.exceptions.HTTPError),
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=2, min=8, max=60),
    reraise=True
)
def _get(url, params=None):
    elapsed = time.time() - _last_call[0]
    if elapsed < THROTTLE_S:
        time.sleep(THROTTLE_S - elapsed)
    resp = requests.get(url, params=params, timeout=20)
    _last_call[0] = time.time()
    if resp.status_code == 429:
        raise requests.exceptions.HTTPError('429 rate limit', response=resp)
    resp.raise_for_status()
    return resp.json()

def get_paginated(endpoint, limit=100):
    """Fetch all pages for an endpoint and return the full list of items."""
    url    = f'{BASE_URL}/{endpoint}.json'
    offset = 0
    items  = []
    while True:
        data  = _get(url, params={'limit': limit, 'offset': offset})
        table = data['MRData']
        total = int(table['total'])

        inner = table.get('RaceTable') or table.get('ConstructorTable') or \
                table.get('DriverTable') or table.get('StandingsTable')
        if not inner:
            break

        for key in ('Races', 'Constructors', 'Drivers', 'StandingsLists'):
            if key in inner:
                items.extend(inner[key])
                break

        offset += limit
        if offset >= total:
            break
    return items

def cache_json(path, data):
    with open(path, 'w') as f:
        json.dump(data, f)

def load_cache(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

print('API client ready.')

API client ready.


In [5]:
import sqlite3

DDL = """
CREATE TABLE IF NOT EXISTS constructors (constructor_id TEXT PRIMARY KEY, name TEXT, nationality TEXT, url TEXT);
CREATE TABLE IF NOT EXISTS drivers (driver_id TEXT PRIMARY KEY, code TEXT, given_name TEXT, family_name TEXT, date_of_birth TEXT, nationality TEXT);
CREATE TABLE IF NOT EXISTS races (race_id INTEGER PRIMARY KEY AUTOINCREMENT, season INTEGER, round INTEGER, race_name TEXT, circuit_id TEXT, circuit_name TEXT, country TEXT, locality TEXT, date TEXT, UNIQUE(season,round));
CREATE TABLE IF NOT EXISTS results (result_id INTEGER PRIMARY KEY AUTOINCREMENT, race_id INTEGER, season INTEGER, round INTEGER, driver_id TEXT, constructor_id TEXT, grid INTEGER, position INTEGER, position_text TEXT, position_order INTEGER, points REAL, laps INTEGER, status TEXT, fastest_lap_rank INTEGER, UNIQUE(race_id,driver_id));
CREATE TABLE IF NOT EXISTS qualifying (qual_id INTEGER PRIMARY KEY AUTOINCREMENT, race_id INTEGER, season INTEGER, round INTEGER, driver_id TEXT, constructor_id TEXT, q1 TEXT, q2 TEXT, q3 TEXT, position INTEGER, UNIQUE(race_id,driver_id));
CREATE TABLE IF NOT EXISTS constructor_standings (cs_id INTEGER PRIMARY KEY AUTOINCREMENT, season INTEGER, round INTEGER, constructor_id TEXT, points REAL, position INTEGER, wins INTEGER, UNIQUE(season,round,constructor_id));
CREATE TABLE IF NOT EXISTS driver_standings (ds_id INTEGER PRIMARY KEY AUTOINCREMENT, season INTEGER, round INTEGER, driver_id TEXT, constructor_id TEXT, points REAL, position INTEGER, wins INTEGER, UNIQUE(season,round,driver_id));
"""
with sqlite3.connect(DB_PATH) as conn:
    conn.executescript(DDL)
print('Schema created.')

Schema created.


In [7]:
# ── Cell 5: Fetch constructors & drivers (metadata)
import pandas as pd
from tqdm import tqdm
SEASONS = list(range(2014, 2025))
for name, endpoint, id_key, fields in [
    ('constructors', 'constructors', 'constructorId',
         ['constructorId','name','nationality','url']),
    ('drivers', 'drivers', 'driverId_', # Note: id_key 'driverId_' not used here, actual key is 'driverId'
         ['driverId','code','givenName','familyName','dateOfBirth','nationality'])
]:
    cache_path = f'{RAW_DIR}/{name}.json'
    raw = load_cache(cache_path) or get_paginated(endpoint)
    cache_json(cache_path, raw)
    if name == 'constructors':
        rows = [{'constructor_id':c['constructorId'],'name':c['name'],'nationality':c.get('nationality'),'url':c.get('url')} for c in raw]
    else:
        # Corrected: Removed extra ']' from 'driverId]'
        rows = [{'driver_id':d['driverId'],'code':d.get('code'),'given_name':d.get('givenName'),'family_name':d.get('familyName'),'date_of_birth':d.get('dateOfBirth'),'nationality':d.get('nationality')} for d in raw]
    with sqlite3.connect(DB_PATH) as conn:
        pd.DataFrame(rows).to_sql(name, conn, if_exists='replace', index=False)
    print(f'Stored {len(rows)} {name}')

Stored 214 constructors
Stored 879 drivers


In [9]:
# ── Cell 6: Race results per season (main loop ~60 min)
import sqlite3, json, os
from tqdm import tqdm

def upsert_race(conn, season, round_, rd):
    c = rd.get('Circuit',{}); l = c.get('Location',{})
    conn.execute('INSERT OR IGNORE INTO races(season,round,race_name,circuit_id,circuit_name,country,locality,date) VALUES(?,?,?,?,?,?,?,?)',
                 (season,round_,rd.get('raceName'),c.get('circuitId'),c.get('circuitName'),l.get('country'),l.get('locality'),rd.get('date')))
    # Fix 1: Changed `around_` to `round_` and Fix 2: Removed `{` from `round=?`
    return conn.execute('SELECT race_id FROM races WHERE season=? AND round=?',(season,round_)).fetchone()[0]

for season in tqdm(SEASONS, desc='Results'):
    cp = f'{RAW_DIR}/results_{season}.json'
    raw = load_cache(cp) or get_paginated(f'{season}/results')
    cache_json(cp, raw)
    with sqlite3.connect(DB_PATH) as conn:
        for race in raw:
            rid = upsert_race(conn,season,int(race['round']),race)
            for r in race.get('Results',[]):
                flr=None
                if 'FastestLap' in r: flr=int(r['FastestLap'].get('rank',0)) or None
                conn.execute('INSERT OR IGNORE INTO results(race_id,season,round,driver_id,constructor_id,grid,position,position_text,position_order,points,laps,status,fastest_lap_rank) VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)',
                             (rid,season,int(race['round']),r['Driver']['driverId'], # Fix 3: Changed 'driverId]' to 'driverId'
                              r['Constructor']['constructorId'],
                              int(r.get('grid',0) or 0),int(r['position']) if r.get('position','').isdigit() else None,
                              r.get('positionText'),int(r.get('positionOrder',0)),float(r.get('points',0)),
                              int(r.get('laps',0)),r.get('status'),flr))
        conn.commit()
print('Results done.')

Results: 100%|██████████| 11/11 [01:23<00:00,  7.63s/it]

Results done.


In [11]:
# ── Cell 7: Qualifying data
for season in tqdm(SEASONS, desc='Qualifying'):
    cp = f'{RAW_DIR}/qualifying_{season}.json'
    raw = load_cache(cp) or get_paginated(f'{season}/qualifying')
    cache_json(cp, raw)
    with sqlite3.connect(DB_PATH) as conn:
        for race in raw:
            round_=int(race['round'])
            row=conn.execute('SELECT race_id FROM races WHERE season=? AND round=?',(season,round_)).fetchone()
            if row:
                for q in race.get('QualifyingResults',[]):
                    conn.execute('INSERT OR IGNORE INTO qualifying(race_id,season,round,driver_id,constructor_id,q1,q2,q3,position) VALUES(?,?,?,?,?,?,?,?,?)',
                                 (row[0],season,round_,q['Driver']['driverId'],q['Constructor']['constructorId'],q.get('Q1'),q.get('Q2'),q.get('Q3'),int(q.get('position',0))))
        conn.commit()
print('Qualifying done.')

Qualifying: 100%|██████████| 11/11 [01:30<00:00,  8.19s/it]

Qualifying done.


In [14]:
# ── Cell 8: Standings
for season in tqdm(SEASONS, desc='CStandings'):
    cp=f'{RAW_DIR}/constructor_standings_{season}.json'
    raw=load_cache(cp) or get_paginated(f'{season}/constructorStandings')
    cache_json(cp,raw)
    with sqlite3.connect(DB_PATH) as conn:
        for sl in raw:
            rnd=int(sl.get('round',0))
            for s in sl.get('ConstructorStandings',[]):
                conn.execute('INSERT OR IGNORE INTO constructor_standings(season,round,constructor_id,points,position,wins) VALUES(?,?,?,?,?,?)',
                             (season,rnd,s['Constructor']['constructorId'],float(s.get('points',0)),int(s.get('position',0)),int(s.get('wins',0))))
        conn.commit()
for season in tqdm(SEASONS, desc='DStandings'):
    cp=f'{RAW_DIR}/driver_standings_{season}.json'
    raw=load_cache(cp) or get_paginated(f'{season}/driverStandings')
    cache_json(cp,raw)
    with sqlite3.connect(DB_PATH) as conn:
        for sl in raw:
            rnd=int(sl.get('round',0))
            for s in sl.get('DriverStandings',[]):
                cid=s['Constructors'][0]['constructorId'] if s.get('Constructors') else None # Corrected 'constructorId]' to 'constructorId'
                conn.execute('INSERT OR IGNORE INTO driver_standings(season,round,driver_id,constructor_id,points,position,wins) VALUES(?,?,?,?,?,?,?)', # Corrected 'around' to 'round' and 'aposition' to 'position'
                             (season,rnd,s['Driver']['driverId'],cid,float(s.get('points',0)),int(s.get('position',0)),int(s.get('wins',0))))
        conn.commit()
print('Standings done.')

DStandings: 100%|██████████| 11/11 [00:17<00:00,  1.61s/it]

Standings done.


In [15]:
# ── Cell 9: Validate & export CSVs
import sqlite3, pandas as pd
tables=['constructors','drivers','races','results','qualifying','constructor_standings','driver_standings']
with sqlite3.connect(DB_PATH) as conn:
    for t in tables:
        df=pd.read_sql(f'SELECT * FROM {t}',conn)
        df.to_csv(f'{EXPORTS_DIR}/{t}.csv',index=False)
        print(f'  {t:<25} {len(df):>6} rows')
print('All CSVs exported to', EXPORTS_DIR)

  constructors                 214 rows
  drivers                      879 rows
  races                        228 rows
  results                     4626 rows
  qualifying                  4610 rows
  constructor_standings        112 rows
  driver_standings             247 rows
All CSVs exported to /content/drive/MyDrive/PitWall/data/exports
